In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 00 · Synthetic Data Generation (LIVE)

> **LIVE DEMO LAB.** Every resource is suffixed `-live` so it lives alongside (never overwrites)
> the matching pre-demo lab. Long-running jobs are submitted but not awaited — pivot to the
> pre-demo lab to show the finished outcome while the live job cooks, then come back to confirm.

A model is only as smart as its training data — and right now you have none, just a knowledge base in markdown. This lab turns that one file into a grounded training set: split `data/acme_health_kb.md` into sections, have `gpt-4o-mini` write grounded Q&A pairs from each, and save them as train / validation JSONL in Azure's fine-tuning format. You walk away with 50–80 grounded pairs in `acme_training_live.jsonl` and `acme_validation_live.jsonl`, ready for Lab 01 — *the model just wrote its own training set.*

**Prereqs:** `pip install -r requirements.txt`, `.env` filled in, and `az login`.

---
## Step 1 — Load configuration

In [ ]:
import os, json, random
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

load_dotenv()

AZURE_OPENAI_ENDPOINT     = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION  = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
GENERATOR_DEPLOYMENT      = os.environ.get('GENERATOR_DEPLOYMENT', 'gpt-4o-mini')
TENANT_ID                 = os.environ.get('AZURE_TENANT_ID')

PAIRS_PER_SECTION = 6
TRAIN_SPLIT       = 0.85
KB_FILE           = Path('data/acme_health_kb.md')
TRAIN_FILE        = Path('data/acme_training_live.jsonl')
VALID_FILE        = Path('data/acme_validation_live.jsonl')

SYSTEM_PROMPT = (
    'You are the Acme Health Member Services assistant. Answer member '
    'questions accurately according to Acme Health\'s official policies, '
    'pharmacy procedures, plan benefits, and the My Health Online portal.'
)

_cred = DefaultAzureCredential(interactive_browser_tenant_id=TENANT_ID) if TENANT_ID else DefaultAzureCredential()

client = AzureOpenAI(
    azure_endpoint          = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
    api_version             = AZURE_OPENAI_API_VERSION,
)

print(f'Endpoint        : {AZURE_OPENAI_ENDPOINT}')
print(f'Generator model : {GENERATOR_DEPLOYMENT}')
print(f'Pairs per section: {PAIRS_PER_SECTION}')
print(f'Train split     : {int(TRAIN_SPLIT * 100)}%')

---
## Step 2 — Split the knowledge base into sections

Each `## ` heading in `acme_health_kb.md` becomes one chunk the generator
focuses on. Smaller chunks give the model less rope to hallucinate.

In [ ]:
kb_text = KB_FILE.read_text(encoding='utf-8')
raw_sections = kb_text.split('\n## ')

sections = []
for i, block in enumerate(raw_sections):
    text = block if i == 0 else '## ' + block
    text = text.strip()
    if len(text) < 200:
        continue
    title = text.splitlines()[0].lstrip('#').strip()
    sections.append({'title': title, 'content': text})

print(f'Loaded {len(sections)} sections:')
for s in sections:
    print(f'   - {s["title"]}')

---
## Step 3 — Generate Q&A pairs from each section

Each section is rewritten into `PAIRS_PER_SECTION` question/answer pairs that
are strictly grounded in the source text — no hallucinated phone numbers,
copays, or policies.

In [ ]:
GENERATION_PROMPT = '''You are a dataset curator creating training examples for the Acme Health Member Services AI assistant.

Given the knowledge base section below, generate exactly {n} distinct question-and-answer pairs.

Rules:
- Every answer must be grounded ONLY in the provided text. Do NOT add information not present.
- Mix question types: direct lookups, "what happens if..." scenarios, how-to questions, and natural caller phrasings.
- Phrase questions the way a Acme Health MEMBER would say them on a call (not corporate-speak).
- Answers should be warm, complete, and factually accurate — what a great Acme agent would say.
- Return ONLY a JSON array, no markdown fences, no commentary. Format:
[
  {{"question": "...", "answer": "..."}},
  ...
]

Knowledge base section:
---
{content}
---'''

all_examples = []
errors = []

for idx, section in enumerate(sections, 1):
    print(f'[{idx}/{len(sections)}] Generating for: {section["title"]}', end=' ')
    try:
        response = client.chat.completions.create(
            model    = GENERATOR_DEPLOYMENT,
            messages = [
                { 'role': 'system', 'content': 'You generate JSON training data. Return only valid JSON arrays.' },
                { 'role': 'user',   'content': GENERATION_PROMPT.format(n=PAIRS_PER_SECTION, content=section['content']) },
            ],
            temperature = 0.7,
            max_tokens  = 2000,
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1]
            if raw.startswith('json'):
                raw = raw[4:]
            raw = raw.strip()
        pairs = json.loads(raw)
        for pair in pairs:
            q = pair.get('question', '').strip()
            a = pair.get('answer', '').strip()
            if q and a:
                all_examples.append({
                    'messages': [
                        { 'role': 'system',    'content': SYSTEM_PROMPT },
                        { 'role': 'user',      'content': q },
                        { 'role': 'assistant', 'content': a },
                    ]
                })
        print(f'-> {len(pairs)} pairs')
    except Exception as e:
        print(f'ERROR: {e}')
        errors.append({'section': section['title'], 'error': str(e)})

print(f'\nTotal examples generated: {len(all_examples)}')
if errors:
    print(f'Sections with errors: {len(errors)}')
    for e in errors:
        print(f'   - {e["section"]}: {e["error"]}')

---
## Step 4 — Validate and deduplicate

In [ ]:
valid_examples = []
seen = set()
dropped_invalid = 0
dropped_dupes = 0

for ex in all_examples:
    msgs = ex.get('messages', [])
    roles = [m.get('role') for m in msgs]
    if roles != ['system', 'user', 'assistant']:
        dropped_invalid += 1
        continue
    if any(not m.get('content', '').strip() for m in msgs):
        dropped_invalid += 1
        continue
    q = msgs[1]['content'].lower().strip()
    if q in seen:
        dropped_dupes += 1
        continue
    seen.add(q)
    valid_examples.append(ex)

print(f'Valid examples   : {len(valid_examples)}')
print(f'Dropped invalid  : {dropped_invalid}')
print(f'Dropped duplicate: {dropped_dupes}')

if len(valid_examples) < 10:
    print('\nWARNING: Azure OpenAI fine-tuning requires at least 10 examples.')
    print('Raise PAIRS_PER_SECTION in Step 1 and re-run Step 3.')

---
## Step 5 — Split, shuffle, save JSONL files

Azure OpenAI requires **UTF-8 BOM** encoding on JSONL files (that's the
`utf-8-sig` codec).

In [ ]:
random.seed(42)
random.shuffle(valid_examples)

split_idx = int(len(valid_examples) * TRAIN_SPLIT)
train = valid_examples[:split_idx]
valid = valid_examples[split_idx:]

def save_jsonl(rows, path):
    path.parent.mkdir(exist_ok=True)
    with open(path, 'w', encoding='utf-8-sig') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

save_jsonl(train, TRAIN_FILE)
save_jsonl(valid, VALID_FILE)

print(f'{TRAIN_FILE} - {len(train)} examples')
print(f'{VALID_FILE} - {len(valid)} examples')
print(f'\nTotal: {len(train) + len(valid)} examples ready for Lab 01.')

---
## Step 6 — Preview a few examples

In [ ]:
for ex in train[:5]:
    msgs = ex['messages']
    print(f'Q: {msgs[1]["content"]}')
    print(f'A: {msgs[2]["content"]}')
    print('-' * 70)

---
## Done

Two files are now sitting in `data/`:

| File | Purpose |
|------|---------|
| `acme_training_live.jsonl`   | Training set for Lab 01 |
| `acme_validation_live.jsonl` | Validation set for Lab 01 |

Open **`01_supervised_fine_tuning.ipynb`** next.